In [1]:
# ============================================================
# CELL 1 — Install & API Key
# ============================================================
# !pip install openai

import os
os.environ["OPENAI_API_KEY"] = ""


# ============================================================
# CELL 2 — Imports & Config
# ============================================================
import time, hashlib, logging, json
from dataclasses import dataclass, field
import openai
import urllib.request
from html.parser import HTMLParser

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[
        logging.FileHandler("agent_trace.log"),
        logging.StreamHandler()
    ]
)
log = logging.getLogger("scrape_agent")

def trace(step: str, data: dict):
    log.info(f"TRACE | {step} | {json.dumps(data, ensure_ascii=False)[:300]}")

CHUNK_SIZE    = 500
CHUNK_OVERLAP = 50
TOP_K         = 5
MODEL         = "gpt-4o"

print("✅ Imports & config ready")


# ============================================================
# CELL 3 — Data Classes
# ============================================================
class TextExtractor(HTMLParser):
    def __init__(self):
        super().__init__()
        self.texts = []
        self.skip_tags = {"script", "style", "nav", "footer", "head"}
        self.current_skip = False

    def handle_starttag(self, tag, attrs):
        if tag in self.skip_tags:
            self.current_skip = True

    def handle_endtag(self, tag):
        if tag in self.skip_tags:
            self.current_skip = False

    def handle_data(self, data):
        if not self.current_skip and data.strip():
            self.texts.append(data.strip())

def fetch_url(url: str) -> str:
    try:
        req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
        with urllib.request.urlopen(req, timeout=10) as r:
            html = r.read().decode("utf-8", errors="ignore")
        parser = TextExtractor()
        parser.feed(html)
        return " ".join(parser.texts)[:8000]
    except Exception as e:
        log.warning(f"URL fetch failed: {e}")
        return ""

@dataclass
class Chunk:
    id: str
    text: str
    source: str
    embedding_key: str = ""

@dataclass
class AgentState:
    query: str
    url: str
    raw_text: str = ""
    chunks: list[Chunk] = field(default_factory=list)
    retrieved: list[Chunk] = field(default_factory=list)
    summary: str = ""
    guardrail_ok: bool = True
    eval_score: int = 0
    eval_feedback: str = ""

print("✅ Data classes ready")


# ============================================================
# CELL 4 — Agent Pipeline Functions
# ============================================================
def mcp_fetch(state: AgentState, client: openai.OpenAI) -> AgentState:
    log.info(f"[1/5] FETCH — {state.url}")
    t0 = time.time()
    scraped = fetch_url(state.url) if state.url else ""
    if len(scraped) > 300:
        state.raw_text = scraped
        trace("fetch", {"method": "scrape", "chars": len(scraped), "latency_s": round(time.time()-t0,2)})
    else:
        resp = client.chat.completions.create(
            model=MODEL,
            max_tokens=2000,
            messages=[{
                "role": "user",
                "content": (
                    f"Provide a comprehensive, factual overview of: {state.query}\n"
                    "Include key facts, recent developments, statistics, and expert insights. "
                    "Be detailed and informative."
                )
            }]
        )
        state.raw_text = resp.choices[0].message.content
        trace("fetch", {"method": "llm_fallback", "chars": len(state.raw_text), "latency_s": round(time.time()-t0,2)})
    return state

def chunk_text(state: AgentState) -> AgentState:
    log.info("[2/5] CHUNKING")
    text = state.raw_text
    chunks = []
    step = CHUNK_SIZE - CHUNK_OVERLAP
    for start in range(0, len(text), step):
        snippet = text[start: start + CHUNK_SIZE]
        if len(snippet) < 50:
            break
        cid = hashlib.md5(snippet.encode()).hexdigest()[:8]
        chunks.append(Chunk(id=cid, text=snippet, source=state.url,
                            embedding_key=snippet.lower()))
    state.chunks = chunks
    trace("chunker", {"n_chunks": len(chunks)})
    return state

def rag_retrieve(state: AgentState) -> AgentState:
    log.info("[3/5] RAG RETRIEVAL")
    query_words = set(state.query.lower().split())
    def score(chunk: Chunk) -> float:
        return sum(chunk.embedding_key.count(w) for w in query_words)
    ranked = sorted(state.chunks, key=score, reverse=True)
    state.retrieved = ranked[:TOP_K]
    trace("rag_retrieve", {"top_k": TOP_K, "scores": [score(c) for c in state.retrieved]})
    return state

BLOCKED_TOPICS = ["weapon", "exploit", "hack", "malware", "illegal"]

def guardrails(state: AgentState) -> AgentState:
    log.info("[4a/5] GUARDRAILS CHECK")
    combined = (state.query + " " + state.raw_text[:500]).lower()
    violations = [t for t in BLOCKED_TOPICS if t in combined]
    if violations:
        state.guardrail_ok = False
        state.summary = f"⛔ Guardrail triggered: topic(s) {violations} blocked."
        trace("guardrails", {"blocked": True, "violations": violations})
    else:
        trace("guardrails", {"blocked": False})
    return state

def llm_summarize(state: AgentState, client: openai.OpenAI) -> AgentState:
    if not state.guardrail_ok:
        return state
    log.info("[4b/5] LLM SUMMARIZATION")
    context = "\n\n---\n\n".join(c.text for c in state.retrieved)
    t0 = time.time()
    resp = client.chat.completions.create(
        model=MODEL,
        max_tokens=1024,
        messages=[
            {
                "role": "system",
                "content": (
                    "You are an expert research summarizer. "
                    "Given retrieved context, produce a well-structured factual summary. "
                    "Format exactly as:\n"
                    "## Key Findings\n- bullet points\n\n"
                    "## Summary\nparagraph\n\n"
                    "## Data Points\n- key stats or facts"
                )
            },
            {
                "role": "user",
                "content": f"Query: {state.query}\n\nContext:\n{context}\n\nProduce the structured summary."
            }
        ]
    )
    state.summary = resp.choices[0].message.content
    trace("summarizer", {"tokens_out": resp.usage.completion_tokens, "latency_s": round(time.time()-t0,2)})
    return state

def llm_judge(state: AgentState, client: openai.OpenAI) -> AgentState:
    if not state.guardrail_ok:
        return state
    log.info("[5/5] LLM-AS-JUDGE EVALUATION")
    t0 = time.time()
    resp = client.chat.completions.create(
        model=MODEL,
        max_tokens=512,
        messages=[
            {
                "role": "system",
                "content": (
                    "You are an impartial evaluator. Score the summary on: "
                    "Relevance (1-5), Completeness (1-5), Accuracy (1-5), Clarity (1-5). "
                    "Respond ONLY as JSON with no extra text: "
                    '{"relevance":N,"completeness":N,"accuracy":N,"clarity":N,"overall":N,"feedback":"..."}'
                )
            },
            {
                "role": "user",
                "content": f"Query: {state.query}\n\nSummary:\n{state.summary}"
            }
        ]
    )
    raw = resp.choices[0].message.content.strip()
    try:
        start = raw.index("{"); end = raw.rindex("}") + 1
        eval_data = json.loads(raw[start:end])
        state.eval_score = eval_data.get("overall", 0)
        state.eval_feedback = eval_data.get("feedback", "")
    except Exception:
        state.eval_score = -1
        state.eval_feedback = raw[:200]
    trace("llm_judge", {"score": state.eval_score, "feedback": state.eval_feedback, "latency_s": round(time.time()-t0,2)})
    return state

def run_agent(query: str, url: str = "") -> AgentState:
    client = openai.OpenAI(api_key=os.environ["OPENAI_API_KEY"])
    state = AgentState(query=query, url=url)
    log.info("=" * 60)
    log.info(f"AGENT START | query='{query}'")
    log.info("=" * 60)
    t_total = time.time()
    for attempt in range(1, 4):
        try:
            state = mcp_fetch(state, client)
            break
        except Exception as e:
            log.warning(f"Fetch attempt {attempt} failed: {e}")
            time.sleep(2 ** attempt)
    state = chunk_text(state)
    state = rag_retrieve(state)
    state = guardrails(state)
    state = llm_summarize(state, client)
    state = llm_judge(state, client)
    total = round(time.time() - t_total, 2)
    log.info(f"AGENT DONE | total_latency={total}s | eval_score={state.eval_score}/5")
    log.info("=" * 60)
    return state

print("✅ All agent functions ready")


# ============================================================
# CELL 5 — Run the Agent
# ============================================================
state = run_agent("latest developments in agentic AI 2026")

print("\n" + "=" * 60)
print("📋 AGENT REPORT")
print("=" * 60)
print(f"Query    : {state.query}")
print(f"Chunks   : {len(state.chunks)} | Retrieved: {len(state.retrieved)}")
print(f"Guardrail: {'✅ PASS' if state.guardrail_ok else '❌ BLOCKED'}")
print("-" * 60)
print(state.summary)
print("-" * 60)
print(f"🧑‍⚖️  Eval Score : {state.eval_score}/5")
print(f"   Feedback  : {state.eval_feedback}")
print("=" * 60)
print("📁 Full trace saved to: agent_trace.log")


# ============================================================
# CELL 6 — View Observability Trace
# ============================================================
with open("agent_trace.log", "r") as f:
    print(f.read())
    


2026-04-15 22:34:55,535 [INFO] ============================================================
2026-04-15 22:34:55,535 [INFO] AGENT START | query='latest developments in agentic AI 2026'
2026-04-15 22:34:55,536 [INFO] ============================================================
2026-04-15 22:34:55,536 [INFO] [1/5] FETCH — 


✅ Imports & config ready
✅ Data classes ready
✅ All agent functions ready


2026-04-15 22:35:05,677 [INFO] HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-04-15 22:35:05,694 [INFO] TRACE | fetch | {"method": "llm_fallback", "chars": 4283, "latency_s": 10.16}
2026-04-15 22:35:05,695 [INFO] [2/5] CHUNKING
2026-04-15 22:35:05,696 [INFO] TRACE | chunker | {"n_chunks": 10}
2026-04-15 22:35:05,696 [INFO] [3/5] RAG RETRIEVAL
2026-04-15 22:35:05,697 [INFO] TRACE | rag_retrieve | {"top_k": 5, "scores": [26, 24, 18, 17, 17]}
2026-04-15 22:35:05,697 [INFO] [4a/5] GUARDRAILS CHECK
2026-04-15 22:35:05,698 [INFO] TRACE | guardrails | {"blocked": false}
2026-04-15 22:35:05,698 [INFO] [4b/5] LLM SUMMARIZATION
2026-04-15 22:35:09,318 [INFO] HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-04-15 22:35:09,321 [INFO] TRACE | summarizer | {"tokens_out": 307, "latency_s": 3.62}
2026-04-15 22:35:09,322 [INFO] [5/5] LLM-AS-JUDGE EVALUATION
2026-04-15 22:35:10,980 [INFO] HTTP Request: POST https://api.openai.com/


📋 AGENT REPORT
Query    : latest developments in agentic AI 2026
Chunks   : 10 | Retrieved: 5
Guardrail: ✅ PASS
------------------------------------------------------------
## Key Findings
- Designing AI systems that align with human values and ethics is a key challenge as they gain more independence.
- A "human-in-the-loop" approach is advocated to ensure AI systems remain aligned with human goals.
- Transparency in AI decision-making is crucial for user trust.
- The AI industry, including agentic AI, is rapidly growing, with significant investment across multiple sectors.

## Summary
Agentic AI, which involves systems capable of making decisions on behalf of users or organizations, is gaining traction across various industries. The primary challenge for developers is ensuring these systems maintain human values and ethical standards as they become more autonomous. A "human-in-the-loop" approach is recommended to keep these AI systems aligned with human intentions. Transparency is ke